In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

docs = [
    "Data analysis is important.",
    "Data wrangling is as important as data analysis.",
    "Data science contains data analysis and data wrangling."
]

vectorizer = TfidfVectorizer(
    lowercase=True,
    token_pattern=r"(?u)\b\w+\b",  # 保留单词
    norm="l2",                     # L2 归一化
    use_idf=True,
    smooth_idf=False               # idf = log(N/df) + 1 的变体会受这个影响
)

X = vectorizer.fit_transform(docs)

df = pd.DataFrame(
    X.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=[f"document {i+1}" for i in range(len(docs))]
)

print(df.round(3))

            analysis    and     as  contains   data  important     is  \
document 1     0.410  0.000  0.000     0.000  0.410      0.576  0.576   
document 2     0.187  0.000  0.786     0.000  0.374      0.263  0.263   
document 3     0.199  0.418  0.000     0.418  0.598      0.000  0.000   

            science  wrangling  
document 1    0.000      0.000  
document 2    0.000      0.263  
document 3    0.418      0.280  


In [1]:
import math
import re
import pandas as pd
from collections import Counter

docs = [
    "Data analysis is important.",
    "Data wrangling is as important as data analysis.",
    "Data science contains data analysis and data wrangling."
]

# 1. 预处理：小写 + 去标点 + 分词
tokenized_docs = []
for doc in docs:
    tokens = re.findall(r"\b\w+\b", doc.lower())
    tokenized_docs.append(tokens)

# 2. 建 vocabulary
vocab = sorted(set(token for doc in tokenized_docs for token in doc))

# 3. 计算 df
N = len(tokenized_docs)
df_counts = {}
for word in vocab:
    df_counts[word] = sum(1 for doc in tokenized_docs if word in doc)

# 4. 计算 idf
idf = {}
for word in vocab:
    idf[word] = math.log(N / df_counts[word])  # 自然对数 ln

# 5. 计算 raw tf-idf
tfidf_vectors = []
for doc in tokenized_docs:
    tf = Counter(doc)
    vec = [tf[word] * idf[word] for word in vocab]
    tfidf_vectors.append(vec)

# 6. L2 归一化
normalized_vectors = []
for vec in tfidf_vectors:
    norm = math.sqrt(sum(x * x for x in vec))
    if norm == 0:
        normalized_vectors.append(vec)
    else:
        normalized_vectors.append([x / norm for x in vec])

# 7. 转成表
df = pd.DataFrame(
    normalized_vectors,
    columns=vocab,
    index=[f"document {i+1}" for i in range(N)]
)

print("IDF:")
for word in vocab:
    print(f"{word}: {idf[word]:.3f}")

print("\nTF-IDF (normalized):")
print(df.round(3))

IDF:
analysis: 0.000
and: 1.099
as: 1.099
contains: 1.099
data: 0.000
important: 0.405
is: 0.405
science: 1.099
wrangling: 0.405

TF-IDF (normalized):
            analysis    and     as  contains  data  important     is  science  \
document 1       0.0  0.000  0.000     0.000   0.0      0.707  0.707    0.000   
document 2       0.0  0.000  0.953     0.000   0.0      0.176  0.176    0.000   
document 3       0.0  0.565  0.000     0.565   0.0      0.000  0.000    0.565   

            wrangling  
document 1      0.000  
document 2      0.176  
document 3      0.208  
